# Phase 3b — Table 6 ablation

Nine training runs answering two questions from the Sentence-BERT paper:

1. **Which pooling?** MEAN vs MAX vs CLS.
2. **What should the classifier see?** The paper's striking finding is that
   `|u-v|` alone (69.78) beats `u` and `v` together (66.04), and that adding
   `u*v` on top of `(u,v,|u-v|)` actually *hurts* — 80.78 down to 80.44.

**Roughly 90 minutes** at 100k pairs per run. Reduced scale is deliberate: the
ablation is about the *ordering* of configurations, not their absolute values,
and nine full-size runs will not fit a free session.

**Safe to re-run after a disconnect.** Completed configurations are read back
from `results/ablation.csv` on Drive and skipped, so a dropout costs one run.


## 1. GPU and Drive


In [ ]:
import torch; assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive; drive.mount('/content/drive')


## 2. Code


In [ ]:
%cd /content
![ -d Echo ] && (cd Echo && git pull -q) || git clone -q https://github.com/ayn-aval/Echo.git
%cd /content/Echo
!pip install -q transformers datasets scipy 2>&1 | tail -1

# Point results/ at Drive so completed runs survive a disconnect.
import pathlib, shutil
drv = pathlib.Path('/content/drive/MyDrive/echo/phase3'); drv.mkdir(parents=True, exist_ok=True)
if (drv/'ablation.csv').exists(): shutil.copy(drv/'ablation.csv', 'results/ablation.csv')
print('resuming from', drv/'ablation.csv' if (drv/'ablation.csv').exists() else 'scratch')


## 3. Run the ablation

Re-run this cell if the session drops. It picks up where it stopped.


In [ ]:
!python -m eval.ablation --model distilroberta-base --pairs 100000 \
                        --out-root /content/drive/MyDrive/echo/phase3/ablation

import shutil, pathlib
shutil.copy('results/ablation.csv', '/content/drive/MyDrive/echo/phase3/ablation.csv')
print('saved to Drive')


## 4. Results

Copy the printed table back to the chat — the two CLAIM lines at the bottom are
the ones that matter.


In [ ]:
import pandas as pd
df = pd.read_csv('results/ablation.csv')
df
